# Qcombo → AMC: Angular-Momentum Coupling Workflow

This notebook demonstrates the full workflow:
1. Use **qcombo** to compute commutators in **m-scheme** (magnetic quantum number basis)
2. Use **AMC** (Angular-Momentum Coupling) to reduce the result to **j-scheme** (coupled angular-momentum basis)

**AMC** is a standalone Python package by Julien Ripoche, Alexander Tichai, and Roland Wirth that performs
automated angular-momentum algebra reduction using Yutsis graph techniques. It handles 3j, 6j, and 9j symbols
and outputs Wigner-Eckart reduced matrix elements in LaTeX.

## Prerequisites

Make sure both packages are installed:
```bash
pip install qcombo amc
```

In [1]:
pip install amc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import qcombo
import os
import sys
import subprocess

## Step 1: Generate .amc file with qcombo

Use `easyCombo()` with the `amcOutput` parameter to generate an AMC input file.
This computes the [1B, 1B] commutator and writes the result in AMC's DSL format.

In [3]:
# Compute [1B, 1B] commutator, output both LaTeX (m-scheme) and AMC input
result = qcombo.easyCombo(
    1, 1,  # left=1-body, right=1-body
    latexOutput="m_scheme_commutator_1B1B.tex",
    amcOutput="m_scheme_commutator_1B1B.amc",
    parallel=False,
    show_process=False
)

LaTeX output saved to: m_scheme_commutator_1B1B.tex
AMC input file saved to: m_scheme_commutator_1B1B.amc


### Inspect the generated .amc file

The AMC input file contains:
- **`declare` statements** — define tensors (mode, diagonal, LaTeX label)
- **Equations** — each contraction body gets an equation of the form:
  `R{body} = coefficient * sum_{dummy_indices}(expression);`

In [4]:
with open("m_scheme_commutator_1B1B.amc", "r") as f:
    print(f.read())

declare G{mode= (1,1),latex ="G" } 
declare H{mode= (1,1),latex ="H" } 
declare R0{mode= (0,0),latex ="R" } 
declare R1{mode= (1,1),latex ="R" } 
declare n {  mode=2, diagonal=true, latex="n"} 
 
# commutator [1B,1B]-0B 
# lambda_1B
R0 = 1*sum_ab((n_a-n_b)*G_ab*H_ba);
 
# commutator [1B,1B]-1B 
# lambda_1B
R1_ab = 1*sum_c(G_ac*H_cb-G_cb*H_ac);
 



## Step 2: Run AMC to get j-scheme result

AMC is a **command-line tool**. Use `sys.executable` to ensure we use the same Python
environment as the current notebook (the venv where `amc` is installed).

Alternatively, you can run it directly in a terminal:
```bash
python -m amc m_scheme_commutator_1B1B.amc -o J_scheme_commutator_1B1B.tex
```
or 
```bash
amc m_scheme_commutator_1B1B.amc -o J_scheme_commutator_1B1B.tex
```

### Available options

| Option | Description |
|--------|-------------|
| `-o, --output` | Output LaTeX file path |
| `--collect-ninejs` | Build 9j-coefficients from products of 6j |
| `--keep-threejs` | Keep 3j-coefficients in output |
| `--wet-convention {wigner,sakurai}` | Wigner-Eckart convention (default: wigner) |
| `-v, --verbose` | Increase verbosity level |

In [5]:
# Use sys.executable to run AMC with the current venv's Python
result = subprocess.run(
    [sys.executable, "-m", "amc",
     "m_scheme_commutator_1B1B.amc",
     "-o", "J_scheme_commutator_1B1B.tex"],
    capture_output=True,
    text=True
)

print("STDOUT:", result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("AMC finished successfully!")

STDOUT: Running...
Equation   1/  2...
Equation   2/  2...
Time elapsed: 0:00:00.001995.

AMC ended successfully!

AMC finished successfully!


## Step 3: View the j-scheme result

AMC outputs a complete LaTeX document with the angular-momentum-coupled result.
This contains Wigner-Eckart reduced matrix elements, 3j/6j/9j symbols, etc.

In [6]:
if os.path.exists("J_scheme_commutator_1B1B.tex"):
    with open("J_scheme_commutator_1B1B.tex", "r") as f:
        print(f.read())
else:
    print("Output file not found. Run the AMC step first.")


\documentclass{scrartcl}

\usepackage{expl3}
\usepackage{xparse}
\usepackage{amsmath}
\usepackage{breqn}

\newcommand{\threej}[3]{\begingroup\setlength{\arraycolsep}{0.2em}\begin{Bmatrix} #1 & #2 & #3 \end{Bmatrix}\endgroup}
\newcommand{\sixj}[6]{\begingroup\setlength{\arraycolsep}{0.2em}\begin{Bmatrix} #1 & #2 & #3 \\ #4 & #5 & #6 \end{Bmatrix}\endgroup}
\newcommand{\ninej}[9]{\begingroup\setlength{\arraycolsep}{0.2em}\begin{Bmatrix} #1 & #2 & #3 \\ #4 & #5 & #6 \\ #7 & #8 & #9 \end{Bmatrix}\endgroup}

\ExplSyntaxOn
\tl_new:N \l__hatfact_tl

\NewDocumentCommand{\hatfact} {m} { \__hatfact_parse:n #1 }

\cs_new:Nn \__hatfact_parse:n { \tl_set:Nn \l__hatfact_main_tl {#1} \__hatfact_hat: }

\cs_generate_variant:Nn \str_set:Nn {Nx}
\cs_new:Nn \__hatfact_hat:
  {
    \str_set:Nx \l_tmpa_str {\l__hatfact_main_tl}
    \str_set:Nn \l_tmpb_str {j}
    \str_if_eq:NNTF \l_tmpa_str \l_tmpb_str {
      \hat{\jmath}
    } {
      \hat \l__hatfact_main_tl
    }
  }
\ExplSyntaxOff

\begin{document}



## Workflow Summary

```
qcombo.easyCombo()                    AMC (command line)
==================                    ==================
                                       
  Wick's theorem   ──>  .amc file  ──>  Yutsis graph reduction
  (m-scheme)              (DSL)         (j-scheme)
       │                                     │
       v                                     v
    .tex file                            .tex file
  (m-scheme,                          (j-scheme,
   occupation numbers)                 Wigner-Eckart,
                                       3j/6j/9j symbols)
```

### Key points

- **qcombo** produces expressions with occupation numbers `n_a`, `n_b` in m-scheme
- **AMC** reduces these to j-scheme with reduced matrix elements `\bar{G}`, `\bar{H}` and angular-momentum coupling symbols
- The `.amc` file is a human-readable DSL — you can edit it manually if needed
- AMC must be run from the **command line**, not inside Python directly

### AMC DSL syntax quick reference

**Tensor declaration:**
```
declare <name>{mode=(<up>,<down>), latex="<label>", [diagonal=true]}
```
- `mode=(n,m)`: n upper indices, m lower indices (a scalar is `(0,0)`)
- `mode=n` is shorthand for `mode=(n,n)`
- `diagonal=true`: marks tensor as diagonal (e.g., occupation number `n`)
- `latex`: how the tensor appears in LaTeX output

**Equation:**
```
<lhs> = <coefficient>*sum_<indices>(<expression>);
```
- Use `_` for subscripts: `G_ab` means tensor G with indices a, b
- `sum_ab(...)` sums over dummy indices a, b
- Comments start with `#`
- Each equation ends with `;`